# 02 — Model (finalna próba, wariant EP)

Łączymy doświadczenia dwóch projektów na **data-rich** zbiorze EP
(14 kan., 128 Hz, **~52k** trialów treningowych — ~56× więcej niż Cap64 z raportu):

- **z raportu Cap64**: porządny preprocessing (detrend + bandpass 1–40 Hz + baseline +
  odrzucanie artefaktów), klasyczny baseline (RandomForest na bandpower + Hjorth),
  EEGNet (Lawhern 2018), uczciwa ewaluacja (macro-F1 obok accuracy, per-class).
- **z projektu EP**: surowy sygnał → 1D CNN, tSNE na embeddingach.

Cała logika siedzi w `eeg_lib.py` (testowalna), notebook to warstwa prezentacji.

**Realny cel:** pewnie pobić losowe 10% i baseline LR/RF, celować w wysokie kilkanaście %.
SOTA dla całego MindBigData (11 klas) to ~31%, a wyniki 90%+ z literatury dotyczą innych
headsetów / zadania *imagined* — nie tego setupu (patrz raport, sek. 6.1).

## 1. Setup

In [1]:
import torch  # MUSI być przed matplotlib (konflikt OpenMP na Windows -> segfault)
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE

import eeg_lib as L

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

/Users/hxwk/eeg_mnist_classifier/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


mps 


## 2. Wczytanie + preprocessing

Kolejność jak w raporcie Cap64 (przeniesiona na EP):
1. odrzucenie kodu `-1` (brak bodźca),
2. **detrend** liniowy → **bandpass 1–40 Hz** (Butterworth zero-phase) → **baseline correction**
   (per trial, więc bez wycieku — robione na całości),
3. **split 80/20** — i dopiero teraz liczymy progi/statystyki **tylko z train**,
4. **odrzucanie artefaktów** (peak-to-peak > 10·MAD lub flatline),
5. **z-score per kanał** ze statystyk train.

In [ ]:
ds = load_dataset("DavidVivancos/MindBigData2022_MNIST_EP")
X_full, y_full = L.df_to_raw(ds["train"].to_pandas())
X_test, y_test = L.df_to_raw(ds["test"].to_pandas())
print("po odrzuceniu -1:", X_full.shape, X_test.shape)

# detrend + bandpass 1-40 Hz + baseline (per trial -> brak wycieku, robimy na całości)
X_full = L.filter_signal(X_full)
X_test = L.filter_signal(X_test)

# split PRZED liczeniem statystyk i progów
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.2, stratify=y_full, random_state=SEED
)

# odrzucanie artefaktów: próg p2p liczony NA TRAIN, stosowany wszędzie
thr = L.reject_threshold(X_train, k=10.0)
mtr = L.reject_mask(X_train, thr); X_train, y_train = X_train[mtr], y_train[mtr]
mva = L.reject_mask(X_val, thr);   X_val, y_val = X_val[mva], y_val[mva]
mte = L.reject_mask(X_test, thr);  X_test, y_test = X_test[mte], y_test[mte]
print(f"odrzucono artefaktów: train {1-mtr.mean():.1%}, val {1-mva.mean():.1%}, test {1-mte.mean():.1%}")

# z-score per kanał — statystyki z train
mean, std = L.zscore_fit(X_train)
X_train = L.zscore_apply(X_train, mean, std)
X_val = L.zscore_apply(X_val, mean, std)
X_test = L.zscore_apply(X_test, mean, std)
print("train", X_train.shape, "val", X_val.shape, "test", X_test.shape)

po odrzuceniu -1: (51895, 14, 256) (12980, 14, 256)
odrzucono artefaktów: train 17.1%, val 16.6%, test 17.4%
train (34416, 14, 256) val (8659, 14, 256) test (10717, 14, 256)


## 3. Sanity check — tiny MLP na 2000 próbkach

Cel: potwierdzić, że dane są poprawnie podane i loss spada (start ~`ln(10)≈2.30`).

In [3]:
idx = np.random.choice(len(X_train), 2000, replace=False)
tiny_loader = L.make_loader(X_train[idx], y_train[idx], batch=64, shuffle=True)
tiny = nn.Sequential(nn.Flatten(), nn.Linear(14 * 256, 64), nn.ReLU(), nn.Linear(64, 10)).to(device)
opt = torch.optim.Adam(tiny.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(3):
    tiny.train()
    for xb, yb in tiny_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = loss_fn(tiny(xb), yb)
        loss.backward()
        opt.step()
    print(f"ep {epoch}  loss={loss.item():.4f}")

/Users/hxwk/eeg_mnist_classifier/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


ep 0  loss=2.2465
ep 1  loss=1.5638
ep 2  loss=1.6061


## 4. Baseline — LogisticRegression + RandomForest

LR na surowych spłaszczonych próbkach (jak w pierwszej wersji) oraz RandomForest na cechach
bandpower (5 pasm, Welch) + Hjorth (3 param.) = 14·8 = 112 cech (podejście z raportu Cap64).

In [4]:
# LogisticRegression na spłaszczonym sygnale
Xtr_flat = X_train.reshape(len(X_train), -1)
Xte_flat = X_test.reshape(len(X_test), -1)
lr = LogisticRegression(max_iter=200).fit(Xtr_flat, y_train)
pred_lr = lr.predict(Xte_flat)
print(f"LR  test acc={accuracy_score(y_test, pred_lr):.4f}  f1={f1_score(y_test, pred_lr, average='macro'):.4f}")

# RandomForest na cechach bandpower + Hjorth
Ftr = L.bandpower_hjorth_features(X_train)
Fte = L.bandpower_hjorth_features(X_test)
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=SEED)
rf.fit(Ftr, y_train)
pred_rf = rf.predict(Fte)
print(f"RF  test acc={accuracy_score(y_test, pred_rf):.4f}  f1={f1_score(y_test, pred_rf, average='macro'):.4f}")

/Users/hxwk/eeg_mnist_classifier/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LR  test acc=0.1336  f1=0.1334
RF  test acc=0.1043  f1=0.1039


## 5. Modele deep — EEGNet1D + EEGNet (raw signal + augmentacja)

Oba kompaktowe, trenują się w minuty na RTX 3060. Mając ~40k trialów po czyszczeniu możemy
docisnąć augmentacją (time-shift, jitter, channel dropout) + weight decay + cosine LR +
early stopping na val macro-F1. Twardy limit 300 s na trening.

In [5]:
train_loader = L.make_loader(X_train, y_train, batch=128, shuffle=True)
val_loader = L.make_loader(X_val, y_val, batch=256)
test_loader = L.make_loader(X_test, y_test, batch=256)
print("batches/epoch:", len(train_loader))

batches/epoch: 269


In [ ]:
torch.manual_seed(SEED)
cnn = L.EEGNet1D().to(device)
print("EEGNet1D params:", sum(p.numel() for p in cnn.parameters()))
cnn, hist_cnn = L.train_model(cnn, train_loader, val_loader, device,
                              epochs=50, time_budget_s=300, log_every=1, use_agument=False, patience=20)

In [6]:
torch.manual_seed(SEED)
egn = L.EEGNet().to(device)
print("EEGNet params:", sum(p.numel() for p in egn.parameters()))
egn, hist_egn = L.train_model(egn, train_loader, val_loader, device,
                              epochs=50, time_budget_s=300, log_every=1, use_augment=False, patience=20)

EEGNet params: 2618


/Users/hxwk/eeg_mnist_classifier/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  ep 01  train_acc=0.104  val_acc=0.122  val_f1=0.107


/Users/hxwk/eeg_mnist_classifier/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  ep 02  train_acc=0.126  val_acc=0.135  val_f1=0.116


...

  ep 48  train_acc=0.184  val_acc=0.178  val_f1=0.159


/Users/hxwk/eeg_mnist_classifier/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  ep 49  train_acc=0.184  val_acc=0.178  val_f1=0.160


/Users/hxwk/eeg_mnist_classifier/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  ep 50  train_acc=0.183  val_acc=0.178  val_f1=0.159


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for h, name in [(hist_cnn, "EEGNet1D"), (hist_egn, "EEGNet")]:
    axes[0].plot(h["train_acc"], label=f"{name} train")
    axes[0].plot(h["val_acc"], "--", label=f"{name} val")
    axes[1].plot(h["val_f1"], label=name)
axes[0].axhline(0.1, color="gray", ls=":", label="losowo 10%")
axes[0].set_title("accuracy"); axes[0].legend(fontsize=8)
axes[1].axhline(0.1, color="gray", ls=":")
axes[1].set_title("val macro-F1"); axes[1].legend(fontsize=8)
plt.tight_layout()

## 6. Ewaluacja na test

In [ ]:
results = {}
for name, p in [("LR", pred_lr), ("RF", pred_rf),
                ("EEGNet1D", L.evaluate(cnn, test_loader, device)[0]),
                ("EEGNet", L.evaluate(egn, test_loader, device)[0])]:
    acc = accuracy_score(y_test, p)
    f1 = f1_score(y_test, p, average="macro")
    results[name] = (acc, f1, p)
    print(f"{name:10s} acc={acc:.4f}  macro-F1={f1:.4f}")

# najlepszy wybieramy wg macro-F1 (accuracy potrafi mylić przy zdegenerowanym klasyfikatorze)
best = max(results, key=lambda k: results[k][1])
pred_best = results[best][2]
print("\nnajlepszy (macro-F1):", best)

In [ ]:
cm = confusion_matrix(y_test, pred_best)
ConfusionMatrixDisplay(cm, display_labels=list(range(10))).plot(cmap="Blues", colorbar=False)
plt.title(f"{best} — confusion matrix (test)")

In [ ]:
per_class = cm.diagonal() / cm.sum(axis=1).clip(min=1)
plt.figure(figsize=(7, 3))
plt.bar(range(10), per_class)
plt.axhline(0.1, color="gray", ls="--", label="losowo 10%")
plt.xticks(range(10)); plt.xlabel("cyfra"); plt.ylabel("accuracy")
plt.title(f"{best} — accuracy per klasa"); plt.legend()

## 7. Analiza błędów — top 5 par

In [ ]:
cm_off = cm.copy(); np.fill_diagonal(cm_off, 0)
pairs = sorted((((i, j), cm_off[i, j]) for i in range(10) for j in range(10) if cm_off[i, j] > 0),
               key=lambda t: -t[1])
print("Najczęstsze pomyłki (true -> pred):")
for (t, p), n in pairs[:5]:
    print(f"  {t} -> {p}: {n}")

## 8. tSNE na embeddingach

Wektor przed głowicą klasyfikującą. Klastry per kolor → sieć nauczyła się separować klasy.

In [ ]:
best_model = {"EEGNet1D": cnn, "EEGNet": egn}.get(best, cnn)
best_model.eval()
embs = []
with torch.no_grad():
    for xb, _ in test_loader:
        embs.append(best_model.embed(xb.to(device)).cpu().numpy())
embs = np.concatenate(embs)

n = min(5000, len(embs))
sel = np.random.choice(len(embs), n, replace=False)
emb2d = TSNE(n_components=2, random_state=SEED).fit_transform(embs[sel])
plt.figure(figsize=(9, 7))
for i in range(10):
    m = y_test[sel] == i
    plt.scatter(emb2d[m, 0], emb2d[m, 1], s=8, alpha=0.4, label=str(i))
plt.legend(); plt.title(f"tSNE embeddingów {best} (test)")

## 9. Porównanie modeli

In [ ]:
rows = [("Losowy baseline", 0.10, np.nan)]
for name in ["LR", "RF", "EEGNet1D", "EEGNet"]:
    acc, f1, _ = results[name]
    rows.append((name, acc, f1))
df_cmp = pd.DataFrame(rows, columns=["model", "test accuracy", "macro-F1"])
display(df_cmp)

fig, ax = plt.subplots(figsize=(7, 3.5))
names = [r[0] for r in rows]; accs = [r[1] for r in rows]
ax.barh(names, accs)
ax.axvline(0.1, ls="--", color="gray")
for i, a in enumerate(accs):
    ax.text(a + 0.003, i, f"{a:.3f}", va="center", fontsize=9)
ax.set_xlabel("test accuracy"); ax.set_title("Porównanie modeli"); ax.invert_yaxis()
plt.tight_layout()